## Pillar 6 — Common Error Message Reference

Quick-lookup cheatsheet for error messages you'll actually see from `ping`, `traceroute`, `nslookup`, and `dig`. This file grows as we cover more commands (`curl`, `ssh`, etc.; `nc` is now covered) later in the pillar — treat it as a living reference, not a one-time lesson.

**How to use this:** when a command spits out an error you don't recognize, check here first before assuming the worst. Many of these look alarming but mean something very specific and ordinary.

### `ping` errors

|Message|What it actually means|Likely cause|Next step|
|---|---|---|---|
|`ping: wiki.corp.local: Name or service not known`|DNS resolution failed **before** any packet was sent — this is not a network test result at all|DNS record doesn't exist, resolver is unreachable, or a typo in the hostname|`dig <host> +short` to isolate the DNS problem|
|`Destination Host Unreachable`|A **router along the path** actively replied saying it has no route to the destination — different from silence|Broken/missing route on a router, or the target is on a disconnected subnet|Check `ip route`; this is a routing problem, not a "host is just slow" problem|
|`Network is unreachable`|**Your own machine** has no route at all for that destination — fails instantly, no packet ever leaves|Missing default route, downed interface, misconfigured VPN|Run `ip route` — look for a missing/wrong default route|
|`100% packet loss` (after 4 attempts, no other message)|Packets were sent, nothing came back — DNS worked fine, this is a real network-layer silence|Firewall dropping ICMP, host down, or (WSL) pinging a gateway that silently ignores ICMP (see `ping` lesson gotcha)|`traceroute -n` the same target to see how far packets actually get|

### `traceroute` errors / symbols

|Message|What it actually means|Likely cause|Next step|
|---|---|---|---|
|`* * *` (mid-path, replies resume after)|Those specific hops don't generate "TTL exceeded" ICMP replies, but still forward traffic fine|Router policy/rate-limiting — not a break|Ignore if the trace completes; only worry if it never resumes|
|`* * *` all the way to the end, trace never completes|Either a real routing dead-end, or the destination's edge silently drops/ignores all probes past a point|Destination-side filtering (common on public sites) vs an actual broken path|Confirm reachability another way (`ping`, `curl`) — if those work, it was filtering, not a break|
|`!H` / `!N` / `!P` after a hop's time|An ICMP "Host/Network/Protocol Unreachable" was received **instead of** a normal reply — this is a deliberate rejection, not a timeout|A firewall or router explicitly refusing the probe at that hop|Different from `* * *`: this hop is reachable and responding, just actively saying no|
|`traceroute: command not found`|Not installed (common on fresh WSL images)|—|`sudo apt install traceroute`|

### `nslookup` / `dig` errors

|Message|What it actually means|Likely cause|Next step|
|---|---|---|---|
|`NXDOMAIN`|The domain/record **does not exist** according to the server that answered|Typo, never created, or expired|Double-check spelling, then confirm with the authoritative nameserver directly|
|`SERVFAIL`|The DNS **server itself** hit an error trying to answer — not "doesn't exist," but "couldn't figure it out"|Misconfigured nameserver, broken DNSSEC chain, upstream failure|Try a different resolver (`@8.8.8.8`) to see if it's server-specific|
|`REFUSED`|The server understood the query but is **configured to refuse it** (e.g., recursion not allowed for your IP)|Querying a nameserver that only answers authoritative queries, not recursive ones|Query a proper recursive resolver instead, or add `NS`/non-recursive framing|
|`;; communications error to X#53: timed out` / `no servers could be reached`|Nothing answered at all — not a DNS-level error, a **transport-level failure** reaching the resolver|The configured DNS server is down, unreachable, or blocked by a firewall|`ping <dns-server-ip>` to check if the resolver itself is even alive|
|`ANSWER: 0` with `status: NOERROR`|The domain exists, but has **no record of the type you asked for**|Asked for `MX` on a domain with no mail setup, or `A` on a CNAME-only/IPv6-only name|Try a different record type (`AAAA`, `CNAME`, `TXT`)|
|`;; UDP setup with 2001:...#53 ... network unreachable`|`dig +trace` picked an **IPv6-only** nameserver and your machine has no IPv6 route out|Common WSL2 quirk — no outbound IPv6|Harmless if `dig` retries with an IPv4 nameserver and succeeds — only worry if the whole trace stops here|

### `nc` errors

|Message|What it actually means|Likely cause|Next step|
|---|---|---|---|
|`Connection to host port [tcp/x] succeeded!`|The TCP handshake completed — path is open and something is listening (not an error, the baseline)|—|If the app still fails, the problem is above TCP (TLS, auth, app logic) — try `curl -v`|
|`nc: connect to host port N (tcp) failed: Connection refused`|Host answered with a TCP RST — it's **alive**, but nothing is listening on that port. **Fast** failure|Service down, wrong port, or service bound to `127.0.0.1` only|On the target: `systemctl status <svc>`, `ss -tlnp` (check port **and** bind address), `journalctl -u <svc>`|
|`nc: connect to host port N (tcp) timed out: Operation now in progress` (hangs until `-w` expires)|Nothing came back — packets are being **dropped**. **Slow** failure, check `real` time with `time`|Firewall / security group silently dropping, host down, broken route|`ping` + `traceroute -n` to narrow down. Do **not** check the service on the target yet — packets may never reach it|
|`nc: connect to host port N (tcp) failed: Network is unreachable`|**Your own machine** has no route for that address — instant, no packet leaves|Often IPv6 on WSL2 (no outbound IPv6 route) when the name resolves to both A and AAAA|Add `-4` to ignore IPv6; `ip route` if it happens on IPv4 too|
|`nc: getaddrinfo ...: Name or service not known`|DNS failed **before** any connection attempt|Typo, missing record, resolver unreachable|`dig <host> +short`|
|`nc -zvu` says `succeeded` (UDP)|**Not proof of anything** — UDP has no handshake; "no ICMP error came back" is all it means|Open **or** filtered, `nc` can't tell|Test the real protocol: `dig @<server> <name>`|

**General pattern worth remembering across all of these:** split every error into one of three buckets —

1. **Resolution failed** (`Name or service not known`, `NXDOMAIN`) — never got an IP to even try.
2. **Transport failed** (`Network is unreachable`, `timed out`, `no servers could be reached`) — got an IP, couldn't reach anything at that address.
3. **Reached something, but it said no** (`Destination Host Unreachable`, `REFUSED`, `Connection refused` from `nc`/`curl`) — a real device answered and actively rejected you.

Each bucket points at a different layer and a different fix — that's the fastest way to triage an unfamiliar error message you haven't memorized yet.